# Fine-Tuning BERT with Hugging Face in README.md

Below is a complete guide formatted for a GitHub `README.md` to fine-tune a pre-trained BERT model for text classification using the Hugging Face `transformers` library. This example uses the IMDB dataset for sentiment analysis.

## Prerequisites

Install the required packages:

In [ ]:
!pip install transformers datasets torch

### 1. Load Pre-trained BERT Model and Tokenizer

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

# Specify the model name
model_name = "bert-base-uncased"

# Load the tokenizer
tokenizer = BertTokenizer.from_pretrained(model_name)

# Load the model with a classification head (2 labels for binary classification)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

### 2. Prepare the Dataset

In [ ]:
from datasets import load_dataset

# Load the IMDB dataset
dataset = load_dataset("imdb")

# Define the tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Tokenize the entire dataset in batches
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Shuffle and select a small subset for training and evaluation
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

### 3. Convert Dataset to PyTorch Format

In [ ]:
# Rename 'label' column to 'labels' as required by Hugging Face models
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Convert to PyTorch format
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

### 4. Set Up DataLoader

In [ ]:
from torch.utils.data import DataLoader

# DataLoader for training with shuffling
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=8)

# DataLoader for evaluation without shuffling
eval_dataloader = DataLoader(eval_dataset, batch_size=8)

### 5. Define Optimizer and Scheduler

In [ ]:
from transformers import AdamW, get_linear_schedule_with_warmup
import torch

# Define the optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Scheduler setup
num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Move model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

### 6. Training Loop

In [ ]:
from tqdm import tqdm

model.train()
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    for batch in tqdm(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    print(f"Loss: {loss.item()}")

### 7. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score

model.eval()
predictions, true_labels = [], []

with torch.no_grad():
    for batch in tqdm(eval_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)
        predictions.extend(preds.cpu().tolist())
        true_labels.extend(batch["labels"].cpu().tolist())

accuracy = accuracy_score(true_labels, predictions)
print(f"Validation Accuracy: {accuracy:.4f}")

### 8. Save the Fine-Tuned Model

In [ ]:
model.save_pretrained("fine_tuned_bert_imdb")
tokenizer.save_pretrained("fine_tuned_bert_imdb")

### 9. Inference (Optional)

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis", model="fine_tuned_bert_imdb", tokenizer="fine_tuned_bert_imdb")
result = classifier("This movie is great!")
print(result)